In [4]:
!pip install medmnist
!pip install torch torchvision scikit-learn matplotlib


In [ ]:
import torch

import torch.nn.functional as F
import torch.optim as optim
import seaborn as sns
import medmnist
from medmnist import PneumoniaMNIST
from torchvision import transforms
from torch.utils.data import DataLoader

from medmnist import PneumoniaMNIST
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import resnet18

# Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

# Datasets
train_dataset = PneumoniaMNIST(split='train', download=True)
test_dataset = PneumoniaMNIST(split='test', download=True)

# Apply transform
train_dataset = [(transform(img), label) for img, label in train_dataset]
test_dataset = [(transform(img), label) for img, label in test_dataset]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)


model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        labels = labels.squeeze().long()

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")
model.eval()



correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        labels = labels.squeeze().long()
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

print("Test Accuracy:", correct/total)


def generate_medical_report(prediction):

    if prediction == 1:
        return """
CHEST X-RAY REPORT

Findings:
Increased lung opacity patterns may be observed.
Radiographic appearance suggests inflammatory consolidation.

Impression:
Findings are consistent with pneumonia.
Clinical correlation is recommended.
"""
    else:
        return """
CHEST X-RAY REPORT

Findings:
Lung fields appear clear.
No visible consolidation or abnormal opacity.

Impression:
No radiographic evidence of pneumonia.
"""

image, label = test_dataset[0]
image = image.unsqueeze(0)

with torch.no_grad():
    output = model(image)
    pred = torch.argmax(output, dim=1).item()

print("Prediction:", pred)
print("True:", label)

print(generate_medical_report(pred))




Epoch 1, Loss: 0.15045120227276473
Epoch 2, Loss: 0.09296928569705956
Epoch 3, Loss: 0.07801699087439054
Epoch 4, Loss: 0.08550638749305084
Epoch 5, Loss: 0.06846804533002747
Test Accuracy: 0.8269230769230769
Prediction: 1
True: [1]

CHEST X-RAY REPORT

Findings:
Increased lung opacity patterns may be observed.
Radiographic appearance suggests inflammatory consolidation.

Impression:
Findings are consistent with pneumonia.
Clinical correlation is recommended.


Image 1
Prediction: 1 | True: [1]

CHEST X-RAY REPORT

Findings:
Increased lung opacity patterns may be observed.
Radiographic appearance suggests inflammatory consolidation.

Impression:
Findings are consistent with pneumonia.
Clinical correlation is recommended.


Image 2
Prediction: 1 | True: [0]

CHEST X-RAY REPORT

Findings:
Increased lung opacity patterns may be observed.
Radiographic appearance suggests inflammatory consolidation.

Impression:
Findings are consistent with pneumonia.
Clinical correlation is recommended.


